# VeriMedia — Deepfake & AI Image Classification Training Notebook
**Authors:** Yogesh Tiwari & Dhwaj Chauhan  
**Department:** Department of Computer Science & Engineering  
**Project:** VeriMedia Client-Side Media Forensics Engine  
**Hardware Target:** Google Colab Free Tier (Nvidia T4 GPU)  

--- 
### Notebook Overview:
1. **Environment Setup & Dependencies** (PyTorch, Torchvision, ONNX, Albumentations)
2. **Dataset Preprocessing & Custom Augmentation Pipeline** (Simulates social media JPEG recompression)
3. **MobileNetV2 Model Architecture Construction** (Quantization-ready classification head)
4. **Model Training & Validation Loop** (Loss curves & Accuracy tracking)
5. **Evaluation Metrics** (Confusion Matrix, Precision, Recall, F1-Score)
6. **ONNX Model Quantization & Export** (Export to `verimedia_mobilenet_v2.onnx` for browser WASM/WebGL execution)

In [ ]:
# Step 1: Install & Import Dependencies
!pip install -q torch torchvision onnx onnxruntime scikit-learn matplotlib seaborn albumentations pillow

import os
import io
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image, ImageFilter
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Using Computation Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

### Step 2: Custom JPEG Augmentation & Dataset Definition

In [ ]:
class JpegRecompressionTransform:
    """Simulates social media recompression (WhatsApp / X / Instagram quality drops)"""
    def __init__(self, quality_min=50, quality_max=95):
        self.q_min = quality_min
        self.q_max = quality_max

    def __call__(self, img):
        if torch.rand(1).item() > 0.4:
            q = int(torch.randint(self.q_min, self.q_max, (1,)).item())
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=q)
            buffer.seek(0)
            img = Image.open(buffer)
        return img

# ImageNet Standardization & Training Transform Pipeline
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    JpegRecompressionTransform(quality_min=50, quality_max=95),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("[✓] Transforms & Preprocessing Pipeline Configured Successfully.")

### Step 3: Model Architecture — Fine-Tuned MobileNetV2

In [ ]:
def build_verimedia_mobilenet():
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    
    # Freeze early layers for transfer learning efficiency
    for param in model.features[:10].parameters():
        param.requires_grad = False
        
    # Custom Binary Classification Head (0 = Real Photo, 1 = Synthetic AI)
    num_ftrs = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(num_ftrs, 128),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(128, 1)
    )
    return model

model = build_verimedia_mobilenet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
print(model.classifier)

### Step 4: Synthetic Training Simulation & Evaluation

In [ ]:
# Simulated dataset generation for notebook demonstration
class SyntheticRealAiDataset(Dataset):
    def __init__(self, num_samples=600, transform=None):
        self.num_samples = num_samples
        self.transform = transform
        self.labels = np.random.randint(0, 2, num_samples)
        
    def __len__(self):
        return self.num_samples
        
    def __getitem__(self, idx):
        # Generate synthetic test tensor image
        img = Image.fromarray(np.uint8(np.random.rand(224, 224, 3) * 255))
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.float32)

train_dataset = SyntheticRealAiDataset(600, train_transforms)
val_dataset = SyntheticRealAiDataset(150, val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("[✓] Dataset Dataloaders Ready (600 Train / 150 Validation Samples).")

In [ ]:
# Training Loop Execution
epochs = 3
print(f"[!] Training MobileNetV2 Classifier for {epochs} Epochs on {device}...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}")

print("[✓] Model Fine-Tuning Completed Successfully.")

### Step 5: ONNX Model Export for Web Deployment ($0 Compute)

In [ ]:
# Export Model to ONNX Format for Client-Side Browser WASM/WebGL Execution
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_filename = "verimedia_mobilenet_v2.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

size_mb = os.path.getsize(onnx_filename) / (1024 * 1024)
print(f"[✓] ONNX Model Exported: {onnx_filename} ({size_mb:.2f} MB)")
print("[✓] Ready to deploy in VeriMedia web app via ONNX Runtime Web!")